In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalogo", "workspace")

catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

TABELA_TICKETS = "silver.tb_tickets"
TABELA_CLIENTES = "silver.tb_clientes"
TABELA_PEDIDOS = "silver.fat_pedidos"
TABELA_PRODUTOS = "silver.dim_produtos"
TABELA_DESTINO_TICKETS = "gold.gold_tickets"

In [0]:
df_tickets = spark.table(TABELA_TICKETS)
df_clientes = spark.table(TABELA_CLIENTES)
df_pedidos = spark.table(TABELA_PEDIDOS)
df_produtos = spark.table(TABELA_PRODUTOS)

In [0]:
#Verificando a premissa de uma linha por ticket
display(
    df_tickets.groupBy("ticket_id")
    .count()
    .filter("count > 1")
)

In [0]:
from pyspark.sql.functions import when, col

df_tickets = df_tickets.select(
    "ticket_id",
    "id_cliente",
    "id_pedido",

    "data_abertura",
    "data_resolucao",

    "tempo_resolucao_horas",

    "agente_suporte",

    "nota_avaliacao",

    when(col("nota_avaliacao") >= 4, "alta")
    .when(col("nota_avaliacao") == 3, "media")
    .when(col("nota_avaliacao") <= 2, "baixa")
    .otherwise("sem_avaliacao")
    .alias("satisfacao_atendimento"),

    "tipo_problema_padronizado"
)

In [0]:
df_clientes = df_clientes.select(
    "id_cliente",

    F.concat_ws(
        " ",
        F.col("nome"),
        F.col("sobrenome")
    ).alias("nome_cliente"),

    F.when(F.col("idade") >= 18, True).otherwise(False).alias("maior_de_idade")
)

In [0]:
df_pedidos = df_pedidos.select(
    "id_produto",

    "id_pedido",
    
)

In [0]:
df_tickets = df_tickets.withColumn(
    "status_ticket",
    F.when(
        F.col("data_resolucao").isNull(),
        "Aberto"
    ).otherwise("Resolvido")
)

In [0]:
df_tickets = df_tickets.withColumn(
    "sla_estourado",
    F.when(
        (
            F.col("status_ticket") == "Aberto"
        ) &
        (
            (
                F.unix_timestamp(F.current_timestamp())
                - F.unix_timestamp("data_abertura")
            ) / 3600 > 48
        ),
        True
    ).otherwise(False)
)

In [0]:
df_tickets = df_tickets.withColumn(
    "data_referencia_calculo",
    F.current_date()
)

In [0]:
df_gold_tickets = (
    df_tickets
    .join(
        df_clientes,
        on="id_cliente",
        how="left"
    )
    .join(
        df_pedidos.select(
            "id_pedido",
            "id_produto"
        ),
        on="id_pedido",
        how="left"
    )
)

In [0]:
df_gold_tickets = (
    df_gold_tickets
    .join(
        df_produtos.select(
            "id_produto",
            "nome_produto"
        ),
        on="id_produto",
        how="left"
    )
)

In [0]:
display(df_gold_tickets)

In [0]:
df_gold_tickets.count()

In [0]:
df_gold_tickets.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO_TICKETS)

print(f"Tabela de tickets salva com sucesso em: {TABELA_DESTINO_TICKETS}")

display(df_gold_tickets.limit(5))

In [0]:
def exportar_csv(df, nome_arquivo):

    caminho_saida = f"/Volumes/workspace/gold/exports/{nome_arquivo}"

    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(caminho_saida)
    )

    print(f"CSV exportado com sucesso em: {caminho_saida}")


In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS gold.exports
""")

In [0]:
exportar_csv(
    df_gold_tickets,
    "gold_tickets_csv"
)